# 05 — 3D Texture Application
Apply generated heightmaps to 3D meshes via UV displacement and export as GLB.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import trimesh
import random

In [ ]:
# ── Create primitive meshes ──
from texture3d.mesh_processor import MeshProcessor

sphere = MeshProcessor.sphere(subdivisions=4)
box    = MeshProcessor.box(extents=(1, 1, 1))

print(f'Sphere: {len(sphere.vertices)} vertices, {len(sphere.faces)} faces')
print(f'Box:    {len(box.vertices)} vertices, {len(box.faces)} faces')

In [ ]:
# ── Generate UV coordinates ──
sphere_uv = MeshProcessor.sphere(subdivisions=4)
sphere_uv.generate_uv(method='sphere')  # or 'xatlas' for better quality

uvs = sphere_uv.get_uv()
print(f'UV shape: {uvs.shape}  |  range: [{uvs.min():.3f}, {uvs.max():.3f}]')

plt.figure(figsize=(5, 5))
plt.scatter(uvs[:, 0], uvs[:, 1], s=0.5, alpha=0.3, c='steelblue')
plt.xlabel('U'); plt.ylabel('V')
plt.title('UV Map — Sphere (Spherical Projection)')
plt.tight_layout(); plt.show()

In [ ]:
# ── Apply heightmap to sphere ──
from texture3d.texture_mapper import HeightmapTextureMapper
from data.heightmap_generator import heightmap_from_path

dtd = Path('../data/dtd/images')
if dtd.exists():
    cats = sorted(dtd.iterdir())
    sample_path = str(random.choice(list(cats[0].glob('*.jpg'))))
else:
    sample_path = 'texture.jpg'

hm = heightmap_from_path(sample_path, sigma=2.0)

mapper = HeightmapTextureMapper(scale_factor=0.08)
displaced = mapper.apply(sphere_uv, hm)

print(f'Displaced mesh: {len(displaced.vertices)} vertices')

# Show displacement magnitude
orig_verts = sphere_uv.vertices
new_verts  = displaced.vertices
disp_mag   = np.linalg.norm(new_verts - orig_verts, axis=1)

print(f'Displacement magnitude: min={disp_mag.min():.4f}  max={disp_mag.max():.4f}  mean={disp_mag.mean():.4f}')

In [ ]:
# ── Visualize before/after ──
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Source texture
axes[0].imshow(Image.open(sample_path))
axes[0].set_title('Source Texture'); axes[0].axis('off')

# Heightmap
axes[1].imshow(np.array(hm), cmap='gray')
axes[1].set_title('Generated Heightmap'); axes[1].axis('off')

# Displacement magnitude on sphere
sc = axes[2].scatter(new_verts[:, 0], new_verts[:, 1], c=disp_mag, cmap='hot', s=1)
plt.colorbar(sc, ax=axes[2], label='Displacement magnitude')
axes[2].set_title('Displaced Sphere (XY projection)')
axes[2].set_aspect('equal')
plt.tight_layout(); plt.show()

In [ ]:
# ── Scale factor comparison ──
from texture3d.texture_mapper import HeightmapTextureMapper

scales = [0.01, 0.05, 0.1, 0.2]
fig, axes = plt.subplots(1, len(scales), figsize=(16, 4))

for ax, scale in zip(axes, scales):
    mp = MeshProcessor.sphere(subdivisions=4)
    mp.generate_uv('sphere')
    mapper = HeightmapTextureMapper(scale_factor=scale)
    disp = mapper.apply(mp, hm)
    dm = np.linalg.norm(disp.vertices - mp.vertices, axis=1)
    sc = ax.scatter(disp.vertices[:, 0], disp.vertices[:, 1], c=dm, cmap='hot', s=0.8)
    ax.set_title(f'scale={scale}'); ax.set_aspect('equal'); ax.axis('off')

plt.suptitle('Displacement Scale Factor Comparison')
plt.tight_layout(); plt.show()

In [ ]:
# ── Export to GLB ──
out_path = '../outputs/textured_sphere.glb'
Path('../outputs').mkdir(exist_ok=True)
displaced.export(out_path)
print(f'Exported: {out_path}')
print('Open in https://gltf.report/ or Blender to view the 3D result.')

In [ ]:
# ── 3DShape2VecSet bridge demo ──
from texture3d.vecset_connector import VecSetMeshLoader, shape_from_image_stub

# If you have a VecSet-generated OBJ, load it like this:
# mp = VecSetMeshLoader.load('path/to/vecset_output.obj')

# Otherwise, use the stub to generate a placeholder shape:
shape_path = shape_from_image_stub(sample_path, output_path='../outputs/placeholder_shape.obj')
print(f'Shape available at: {shape_path}')